In [1]:
!pip install fair-esm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 4.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

In [3]:
# Dataset Class with data augmentation
class PeptideDataset(Dataset):
    def __init__(self, csv_file, augment=False):
        df = pd.read_csv(csv_file)
        self.sequences = df['sequence'].astype(str).tolist()
        self.labels = df['FRS'].tolist()
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]

        # Simple data augmentation - reverse sequence
        if self.augment and torch.rand(1).item() > 0.5:
            seq = seq[::-1]

        return seq, label

In [4]:
# Main Execution
if __name__ == "__main__":
    # Initialize with data augmentation for training
    full_dataset = PeptideDataset("01_AO_db_original_1.csv", augment=True)

    # Stratified split
    from sklearn.model_selection import train_test_split
    indices = list(range(len(full_dataset)))
    labels = [full_dataset[i][1] for i in indices]
    train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)

    train_dataset = torch.utils.data.Subset(full_dataset, train_idx)
    val_dataset = torch.utils.data.Subset(
        PeptideDataset("01_AO_db_original_1.csv", augment=False),
        val_idx
    )

In [5]:
# Extract sequences and labels for train_dataset
train_sequences = [full_dataset.sequences[i] for i in train_idx]
train_labels = [full_dataset.labels[i] for i in train_idx]

# Create a DataFrame for the training set and save to CSV
train_df = pd.DataFrame({'sequence': train_sequences, 'FRS': train_labels})
train_df.to_csv('train_dataset.csv', index=False)
print("train_dataset.csv created successfully.")

# Extract sequences and labels for val_dataset
# Note: Assuming val_dataset is also derived from the same full_dataset as train_dataset
val_sequences = [full_dataset.sequences[i] for i in val_idx]
val_labels = [full_dataset.labels[i] for i in val_idx]

# Create a DataFrame for the validation set and save to CSV
val_df = pd.DataFrame({'sequence': val_sequences, 'FRS': val_labels})
val_df.to_csv('val_dataset.csv', index=False)
print("val_dataset.csv created successfully.")

train_dataset.csv created successfully.
val_dataset.csv created successfully.


In [6]:
train = pd.read_csv("train_dataset.csv")
val = pd.read_csv("val_dataset.csv")

train["sequence"] = train["sequence"].str.strip().str.upper()
val["sequence"] = val["sequence"].str.strip().str.upper()

print("Train size:", len(train))
print("Validation size:", len(val))

train_set = set(train['sequence'])
val_set = set(val['sequence'])

exact_overlap = train_set.intersection(val_set)

print("Exact duplicate sequences:", len(exact_overlap))
print("Leakage rate:", len(exact_overlap)/len(val))

Train size: 731
Validation size: 183
Exact duplicate sequences: 0
Leakage rate: 0.0


In [7]:
def write_fasta(df, filename):
    with open(filename, "w") as f:
        for i, seq in enumerate(df.sequence):
            f.write(f">seq_{i}\n{seq}\n")

write_fasta(train, "train.fasta")
write_fasta(val, "val.fasta")

In [8]:
import getpass
import os

password = getpass.getpass()  # This prompts for the password in the notebook output area or terminal
#command = "sudo -S apt-get update" # Replace 'apt-get update' with your desired command
command = "sudo -S apt-get install cd-hit"

# Use os.system to run the command
os.system(f'echo {password} | {command}')

··········


0

In [9]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.1 MB/s eta 0:00:00


In [10]:
!cat train.fasta val.fasta > combined.fasta
!cd-hit -i combined.fasta -o combined_90 -c 0.9 -n 5

Program: CD-HIT, V4.8.1 (+OpenMP), Aug 20 2021, 08:39:56
Command: cd-hit -i combined.fasta -o combined_90 -c 0.9 -n 5

Started: Mon Mar 30 21:18:20 2026
                            Output                              
----------------------------------------------------------------
total seq: 81
longest and shortest : 36 and 11
Total letters: 1287
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 0M
Buffer          : 1 X 10M = 10M
Table           : 1 X 65M = 65M
Miscellaneous   : 0M
Total           : 75M

Table limit with the given memory limit:
Max number of representatives: 4000000
Max number of word counting entries: 90517868

comparing sequences from          0  to         81

       81  finished         76  clusters

Approximated maximum memory consumption: 75M
writing new database
writing clustering information
program completed !

Total CPU time 0.07


In [11]:
def parse_clusters(clstr_file):
    clusters = []
    current = []
    with open(clstr_file) as f:
        for line in f:
            if line.startswith(">Cluster"):
                if current:
                    clusters.append(current)
                current = []
            else:
                current.append(line.strip())
        clusters.append(current)
    return clusters

clusters = parse_clusters("combined_90.clstr")

leak_clusters = 0
for c in clusters:
    train_flag = any("train" in x for x in c)
    val_flag = any("val" in x for x in c)
    if train_flag and val_flag:
        leak_clusters += 1

print("Clusters containing both train and val:", leak_clusters)

Clusters containing both train and val: 0


In [12]:
from itertools import product
import numpy as np

def kmers(seq, k=6):
    return set(seq[i:i+k] for i in range(len(seq)-k+1))

train_k = [kmers(s,6) for s in train.sequence]
val_k = [kmers(s,6) for s in val.sequence]

def jaccard(a,b):
    union_len = len(a | b)
    if union_len == 0:
        return 0.0  # Or handle as appropriate, e.g., NaN if truly undefined
    return len(a & b) / union_len

high_similarity_pairs = 0

for i, vk in enumerate(val_k):
    for tk in train_k:
        if jaccard(vk, tk) > 0.8:
            high_similarity_pairs += 1
            break

print("Val sequences with high motif overlap:", high_similarity_pairs)

Val sequences with high motif overlap: 0


In [13]:
import torch
import esm
from sklearn.metrics.pairwise import cosine_similarity

model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()

def compute_embeddings(seqs):
    data = [(str(i), s) for i,s in enumerate(seqs)]
    _, _, tokens = batch_converter(data)
    with torch.no_grad():
        out = model(tokens, repr_layers=[6])
    reps = out["representations"][6]
    return reps.mean(1).numpy()

train_emb = compute_embeddings(train.sequence.tolist())
val_emb = compute_embeddings(val.sequence.tolist())

cos_sim = cosine_similarity(val_emb, train_emb)

semantic_leak = np.sum(np.max(cos_sim, axis=1) > 0.95)

print("Val sequences with cosine similarity > 0.95:", semantic_leak)

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D-contact-regression.pt
Val sequences with cosine similarity > 0.95: 182


In [14]:
import torch
import esm
import numpy as np
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

device = "cpu"

model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()

model = model.to(device)
model.eval()

############################################################
# Memory-safe embedding computation
############################################################

def compute_embeddings(seqs, batch_size=32):

    embeddings = []

    for i in tqdm(range(0, len(seqs), batch_size)):

        batch_seqs = seqs[i:i+batch_size]
        data = [(str(j), s) for j,s in enumerate(batch_seqs)]

        _, _, tokens = batch_converter(data)
        tokens = tokens.to(device)

        with torch.no_grad():
            out = model(tokens, repr_layers=[6])

        reps = out["representations"][6]

        for j, seq in enumerate(batch_seqs):
            emb = reps[j, 1:len(seq)+1].mean(0).cpu().numpy()
            embeddings.append(emb)

        del tokens, out, reps
        torch.cuda.empty_cache()

    return np.vstack(embeddings)

############################################################
# Compute embeddings safely
############################################################

train_emb = compute_embeddings(train.sequence.tolist(), batch_size=32)
val_emb = compute_embeddings(val.sequence.tolist(), batch_size=32)

print("Train embeddings:", train_emb.shape)
print("Val embeddings:", val_emb.shape)

############################################################
# Chunked cosine similarity (avoids NxM matrix)
############################################################

threshold = 0.95
semantic_leak = 0

chunk_size = 200

for i in tqdm(range(0, len(val_emb), chunk_size)):

    val_chunk = val_emb[i:i+chunk_size]

    sims = cosine_similarity(val_chunk, train_emb)

    max_sim = sims.max(axis=1)

    semantic_leak += np.sum(max_sim > threshold)

    del sims

print("Val sequences with cosine similarity > 0.95:", semantic_leak)

100%|██████████| 6/6 [00:03<00:00,  1.92it/s]


Train embeddings: (731, 320)
Val embeddings: (183, 320)


100%|██████████| 1/1 [00:00<00:00, 24.60it/s]

Val sequences with cosine similarity > 0.95: 181


In [15]:
from sklearn.neighbors import NearestNeighbors

combined_emb = np.vstack([train_emb, val_emb])
labels = ["train"]*len(train_emb) + ["val"]*len(val_emb)

nbrs = NearestNeighbors(n_neighbors=5, metric="cosine").fit(combined_emb)
distances, indices = nbrs.kneighbors(combined_emb)

cross_neighbors = 0
for i, neigh in enumerate(indices):
    for n in neigh[1:]:
        if labels[i] != labels[n]:
            cross_neighbors += 1
            break

print("Sequences with cross-split nearest neighbors:", cross_neighbors)

Sequences with cross-split nearest neighbors: 631


In [16]:
print("\n==== DATA LEAKAGE SUMMARY ====")
print("Exact duplicates:", len(exact_overlap))
print("CD-HIT cluster leakage:", leak_clusters)
print("High k-mer leakage:", high_similarity_pairs)
print("High semantic leakage:", semantic_leak)
print("Cross-nearest-neighbor leakage:", cross_neighbors)


==== DATA LEAKAGE SUMMARY ====
Exact duplicates: 0
CD-HIT cluster leakage: 0
High k-mer leakage: 0
High semantic leakage: 181
Cross-nearest-neighbor leakage: 631
